In [ ]:
import pytools as pt

filename = "bulk_compressed.0000015.vlsv"
f = pt.vlsvfile.VlsvReader(filename)
f.list()



tag = PARAMETER
    time
    dt
    timestep
    fieldSolverSubcycles
    fileIndex
    xmin
    xmax
    ymin
    ymax
    zmin
    zmax
    xcells_ini
    ycells_ini
    zcells_ini
    velocity_block_width
    version
    numWritingRanks
    VDF_BYTE_SIZE
    COMPRESSION
tag = VARIABLE
    CellID
    fg_b
    fg_e
    proton/vg_blocks
    proton/vg_rho
    proton/vg_v
    vg_f_saved
    vg_rhom
    vg_rhoq
    vg_v


In [6]:
# Locate proton BLOCKVARIABLE and BYTESPERCELL offsets from the XML footer, read the blob,
# parse HermSpectrum structs using the helper in analysator.pyVlsv.serializers.
import re
from pathlib import Path
from analysator.pyVlsv import serializers

filename = "bulk_compressed.0000015.vlsv"
data = Path(filename).read_bytes()
txt = data.decode('utf-8', errors='ignore')
m = re.search(r'<BLOCKVARIABLE[^>]*name="proton"[^>]*>(\d+)</BLOCKVARIABLE>', txt)
if not m:
    print('Could not find proton BLOCKVARIABLE tag in XML footer')
else:
    blockvar_off = int(m.group(1))
    print('BLOCKVARIABLE offset for proton:', blockvar_off)
    # BYTESPERCELL for proton (how many bytes are stored for the proton cell blob)
    m2 = re.search(r'<BYTESPERCELL[^>]*name="proton"[^>]*>(\d+)</BYTESPERCELL>', txt)
    bytes_per_cell = int(m2.group(1)) if m2 else None
    print('BYTESPERCELL for proton:', bytes_per_cell)
    # Read the blob starting at the blockvariable offset
    blob = data[blockvar_off:blockvar_off + (bytes_per_cell or 0)] if bytes_per_cell else data[blockvar_off:]
    parsed_list, consumed = serializers.parse_herm_spectrum_sequence(blob, size_t_fmt='Q', real_fmt='d')
    print('Parsed HermSpectrum count:', len(parsed_list), 'bytes consumed:', consumed)
    for i, p in enumerate(parsed_list):
        print(f'--- HermSpectrum #{i} ---')
        print('N_hermite_harmonic:', p['N_hermite_harmonic'])
        print('vth:', p['vth'])
        print('u:', p['u'])
        print('shape:', p['shape'])
        print('HermSpectrum length:', p['HermSpectrum'].size)
        print('HermSpectrum sample (first 10):', p['HermSpectrum'].ravel()[:10].tolist())


BLOCKVARIABLE offset for proton: 9214
BYTESPERCELL for proton: 4322
Parsed HermSpectrum count: 1 bytes consumed: 600
--- HermSpectrum #0 ---
N_hermite_harmonic: 5
vth: 321033.1875
u: [ 317968.4  -194779.61   98205.35]
shape: (79, 63, 47)
HermSpectrum length: 125
HermSpectrum sample (first 10): [-13412140032.0, -57601660.0, -630996544.0, nan, 166766592.0, -7715976704.0, 8750555.0, 96364384.0, nan, -40669704.0]
